# Comparing HFO event types and recording modality across Omni-iEEG

Split out of `raw_data_visualization.ipynb`: that notebook is a deep dive on
one example recording; this one scans every processed recording for
doctor-annotated HFO events and compares waveforms across event types and
recording technologies. Self-contained -- only depends on the processed
`.h5` files, not on the single-recording exploration above.

In [ ]:
import os

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import resample
from torch_brain.data import Data

BIDS_ROOT = "/capstor/scratch/cscs/davalos/data/processed/omni_ieeg"

# Crop applied to the cached 224x224 image, plus the random time shift that is
# PyHFO's only train-time augmentation (kept here only for event_length/fs,
# which get_event_waveform needs -- the actual PyHFO image preprocessing
# lives in raw_data_visualization.ipynb's section 4).
PREPROCESS = dict(
    image_size=224,
    fs=1000,
    freq_range_hz=[10, 500],
    event_length=1000,
    selected_window_size_ms=285,
    selected_freq_range_hz=[10, 290],
    random_shift_ms=45,
)


def get_event_waveform(
    signal, timestamps, channel_idx, event_start_s, event_end_s, preprocess=PREPROCESS
):
    """Extract the `event_length` ms window centered on an HFO event and resample
    it to `fs` Hz, matching PyHFO's own extraction (`extract_waveforms` +
    `concate_edf(..., resample=...)` in Omni-iEEG)."""
    center_s = 0.5 * (event_start_s + event_end_s)
    half_len_s = preprocess["event_length"] / 1000 / 2
    keep = (timestamps >= center_s - half_len_s) & (timestamps < center_s + half_len_s)
    trace = signal[keep, channel_idx].astype(np.float64)
    n_target = int(preprocess["event_length"] / 1000 * preprocess["fs"])
    return resample(trace, n_target)

### 5. Comparing HFO event types and recording modality

Compares annotated HFO events across:
- **event type**: artifact (doctor-flagged false positive), spHFO (co-occurs with a spike), isolated HFO,
- **recording technology**: ECoG vs SEEG (`device.recording_tech`, one value per recording).

Events differ hugely in raw amplitude across patients/channels/modalities, so plotting many raw waveforms
superposed on one axis mostly produces unreadable clutter. This notebook uses **small multiples**
(a grid, one panel per event) as the default view for browsing many examples, and a separate
**normalized overlay** (each waveform z-scored, plotted at low alpha, with the group mean in bold) as the
"superposed" view for comparing average shape across a group -- normalizing is what makes that superposition
actually legible.

As more patients are added to `data/raw/omni_ieeg`, re-run `pipelines/omni_ieeg/pipeline.py` (or `brainsets prepare`,
see the README) and this section will automatically pick up any newly processed `.h5` file that has annotations.

In [ ]:
import pandas as pd


def scan_annotated_recordings(bids_root):
    """List every recording under `bids_root` with its HFO-annotation count,
    recording technology, and dataset -- read directly via h5py so this stays
    cheap even as more (mostly un-annotated) recordings are added.

    `channel`/`detector`/`artifact`/`spike` are only present on `annotations`
    once doctor-labeled HFO events were actually merged in (see
    `_build_annotations` in omni_ieeg.py) -- a recording with only raw
    MNE-embedded annotations (no HFO merge) has none of those fields, so
    counting any non-zero `annotations/start` here would wrongly flag it as
    HFO-annotated and crash `collect_hfo_examples` downstream on ann.channel.
    """
    rows = []
    for fname in sorted(os.listdir(bids_root)):
        if not fname.endswith(".h5"):
            continue
        with h5py.File(os.path.join(bids_root, fname), "r") as f:
            ann = f["annotations"]
            if "channel" in ann:
                channels = ann["channel"].asstr()[:]
                n_ann = int((channels != "").sum())
            else:
                n_ann = 0
            rows.append(
                dict(
                    file=fname,
                    n_annotations=n_ann,
                    recording_tech=f["device"].attrs.get("recording_tech"),
                    dataset_name=f.attrs.get("dataset_name"),
                    fs=f["ieeg"].attrs.get("sampling_rate"),
                )
            )
    return pd.DataFrame(rows)


recordings = scan_annotated_recordings(BIDS_ROOT)
print(recordings)
print("\nannotated events per recording technology:")
print(recordings.groupby("recording_tech")["n_annotations"].sum())

In [ ]:
def collect_hfo_examples(bids_root, recordings, max_events_per_type=15, preprocess=PREPROCESS):
    """Extract up to `max_events_per_type` annotated HFO waveforms per
    (recording, event_type), sampled per type rather than uniformly over all
    events -- artifacts are rare (~1% here), so a blind uniform subsample
    would silently drop them from the comparison."""
    examples = []
    annotated = recordings[recordings["n_annotations"] > 0]
    for _, row in annotated.iterrows():
        with h5py.File(os.path.join(bids_root, row["file"]), "r") as f:
            file_data = Data.from_hdf5(f, lazy=False)
        ch_names = file_data.channels.id.astype(str)
        sig = file_data.ieeg.signal
        ts = file_data.ieeg.timestamps
        ann = file_data.annotations
        ann_channel = ann.channel.astype(str)
        ann_detector = ann.detector.astype(str)

        event_types = np.where(
            np.asarray(ann.artifact) == 1,
            "artifact",
            np.where(np.asarray(ann.spike) == 1, "spHFO", "isolated HFO"),
        )
        for event_type in np.unique(event_types):
            type_idx = np.where(event_types == event_type)[0]
            sample_idx = type_idx[np.linspace(0, len(type_idx) - 1, min(len(type_idx), max_events_per_type)).astype(int)]
            print(f"{row['file']}: sampling {len(sample_idx)}/{len(type_idx)} '{event_type}' events")

            for i in sample_idx:
                ch_idx = np.where(ch_names == ann_channel[i])[0]
                if len(ch_idx) == 0:
                    continue
                waveform = get_event_waveform(sig, ts, ch_idx[0], ann.start[i], ann.end[i], preprocess)
                examples.append(
                    dict(
                        file=row["file"],
                        recording_tech=row["recording_tech"],
                        dataset_name=row["dataset_name"],
                        channel=ann_channel[i],
                        detector=ann_detector[i],
                        event_type=event_type,
                        waveform=waveform,
                    )
                )
    return examples


examples = collect_hfo_examples(BIDS_ROOT, recordings, max_events_per_type=15)
print(f"\ncollected {len(examples)} example events total")

In [ ]:
def plot_event_grid(examples, group_key, n_per_group=12, ncols=6, preprocess=PREPROCESS):
    """Small multiples of raw waveforms, one panel per event, grouped by
    `group_key` (e.g. "event_type", "recording_tech"). Always prints how many
    events exist per group vs. how many are actually plotted."""
    groups = {}
    for ex in examples:
        groups.setdefault(ex[group_key], []).append(ex)

    for group_name, group_examples in groups.items():
        shown = group_examples[:n_per_group]
        t_ms = np.linspace(0, preprocess["event_length"], len(shown[0]["waveform"]))
        nrows = int(np.ceil(len(shown) / ncols))
        fig, axes = plt.subplots(nrows, ncols, figsize=(2.2 * ncols, 1.6 * nrows), sharex=True)
        axes = np.atleast_1d(axes).ravel()
        for ax, ex in zip(axes, shown):
            ax.plot(t_ms, ex["waveform"], color="black", linewidth=0.6)
            ax.set_title(ex["channel"], fontsize=7)
            ax.set_xticks([])
            ax.set_yticks([])
        for ax in axes[len(shown):]:
            ax.axis("off")
        fig.suptitle(f"{group_key} = {group_name}  ({len(shown)}/{len(group_examples)} events shown)")
        fig.tight_layout()
        fig.show()


plot_event_grid(examples, group_key="event_type")
plot_event_grid(examples, group_key="recording_tech")

In [ ]:
def plot_group_overlay(examples, group_key, preprocess=PREPROCESS):
    """Normalized (z-scored) overlay of every event's waveform per group, with
    the group mean in bold -- a superposition view that stays legible across
    events of very different raw amplitude."""
    groups = {}
    for ex in examples:
        groups.setdefault(ex[group_key], []).append(ex)

    fig, axes = plt.subplots(1, len(groups), figsize=(5 * len(groups), 3.5), sharey=True)
    axes = np.atleast_1d(axes)
    for ax, (group_name, group_examples) in zip(axes, groups.items()):
        traces = np.stack(
            [
                (ex["waveform"] - ex["waveform"].mean()) / (ex["waveform"].std() + 1e-9)
                for ex in group_examples
            ]
        )
        t_ms = np.linspace(0, preprocess["event_length"], traces.shape[1])
        ax.plot(t_ms, traces.T, color="steelblue", alpha=0.15, linewidth=0.7)
        ax.plot(t_ms, traces.mean(axis=0), color="black", linewidth=2, label="mean")
        ax.set_title(f"{group_key} = {group_name} (n={len(group_examples)})")
        ax.set_xlabel("Time (ms)")
        ax.legend()
    axes[0].set_ylabel("z-scored amplitude")
    fig.tight_layout()
    return fig


fig = plot_group_overlay(examples, group_key="event_type")
fig.show()
fig = plot_group_overlay(examples, group_key="recording_tech")
fig.show()